# Modellering af patientoplevelsesvurderinger på tværs af faciliteter og specialer med PROC FACTMAC

## Sammenfatning for ledelsen

Et flersite-sundhedsvæsen ønsker at lære **interaktionsstrukturen** mellem behandlingsfaciliteter
og kliniske specialer, der driver patienttilfredshedsscorer, så det kan opdage facilitets-/
specialekombinationer, der under- eller overpræsterer. Denne notebook tilpasser en
faktoriseringsmaskine med **PROC FACTMAC**, hvor `facility` og `specialty` behandles som de to
nominelle feature-felter og tilfredshedsscoren fra 1-10 som det intervalskalerede mål, og
evaluerer derefter de rekonstruerede vurderinger.

På de 100 simulerede besøg når modellen en **trænings-R-Square på 0,516** (gennemsnitlig absolut
fejl 0,95 vurderingspoint, RASE 1,25), og dens forudsagte cellegennemsnit adskiller tydeligt de
stærkeste og svageste kombinationer — `Nordklinikken`/`Kardiologi` i toppen versus
`Sydklinikken`/`Kardiologi` i bunden — hvilket genfinder den indlejrede interaktion i stedet for
at falde sammen til det samlede gennemsnit på omkring 6,8.

## Datakilder

Alle data genereres inline af et DATA-trin (`call streaminit(20240531)` + `rand()`), så
notebooken er fuldt selvstændig — ingen eksterne filer eller netværksadgang. Dette miljø kører
ulicenseret, hvilket begrænser output til 100 observationer, så designet er dimensioneret til at
demonstrere faktoriseringsmaskinen **inden for 100 besøg**: tre faciliteter krydset med to
specialer (seks celler, i gennemsnit omkring 17 besøg hver — nok signal pr. celle til at
stokastisk gradientnedstigning kan genfinde strukturen).

**WORK.ENCOUNTERS** — 100 syntetiske patientbesøg (én række pr. besøg).

| Variabel | Type | Rolle | Beskrivelse |
|----------|------|-------|-------------|
| `facility` | char(20) | Input (nominel) | Behandlingssted — `Nordklinikken`, `Centralklinikken` eller `Sydklinikken` |
| `specialty` | char(14) | Input (nominel) | Klinisk speciale — `Kardiologi` eller `Onkologi` |
| `satisfaction` | num | Mål (interval) | Patientoplevelsesvurdering på en 1-10-skala, genereret fra en facilitetsbias + specialebias + et reelt facilitet×speciale-interaktionsled + Gaussisk støj, derefter klippet til [1,10] og afrundet til 0,1 |

Det latente design indlejrer velseparerede facilitets- og specialebias plus en stærk
interaktionseffekt, så en faktoriseringsmaskine bør genfinde struktur, som et facilitets- eller
specialegennemsnit alene ville overse.

# Modellering af patientoplevelsesvurderinger med PROC FACTMAC

Flersite-sundhedsvæsener indsamler patienttilfredshedsscorer på tværs af mange **faciliteter**
og kliniske **specialer**. Gennemsnitlige scorer efter facilitet eller efter speciale alene
skjuler det interessante signal: et speciale kan skinne på ét site og kæmpe på et andet. En
**faktoriseringsmaskine** fanger netop denne slags parvise interaktion ved at lære latente
faktorer for hver facilitet og hvert speciale og derefter modellere hver vurdering som et
globalt gennemsnit plus per-feature-effekter plus deres interaktion.

`PROC FACTMAC` tilpasser denne model med stokastisk gradientnedstigning. I denne notebook vil vi:

1. Generere en syntetisk besøgstabel med en indlejret facilitet x speciale-interaktion,
   dimensioneret til 100-observationsmiljøet.
2. Tilpasse en faktoriseringsmaskine med `PROC FACTMAC`, med anmodning om scorede forudsigelser
   og en iterationshistorik.
3. Evaluere rekonstruktionsfejlen og fremhæve de facilitet x speciale-kombinationer, som
   modellen markerer som stærkest og svagest.

## Trin 1 - Generér syntetiske patientoplevelsesdata

Vi simulerer 100 besøg på tværs af 3 faciliteter og 2 specialer. Hver facilitet og hvert speciale
bærer en skjult bias, og vi tilføjer et reelt **interaktions**-led, så visse facilitets-/
specialekombinationer vurderes højere eller lavere, end deres individuelle bias ville forudsige.
Gaussisk støj (standardafvigelse 0,25) gør vurderingerne realistiske, og vi klipper til
1-10-skalaen og runder til én decimal. `call streaminit`-seedet gør data reproducerbare.

In [1]:
data encounters;
    CALL streaminit(20240531);
    LÆNGDE facility $20 specialty $14;

    /* Navngivne faciliteter og kliniske serviceområder */
    TABEL facs[3] $20 _temporary_ (
        'Nordklinikken' 'Centralklinikken' 'Sydklinikken');
    TABEL specs[2] $14 _temporary_ (
        'Kardiologi' 'Onkologi');

    /* Skjulte per-facilitet og per-speciale vurderingsbias */
    TABEL f_bias[3] _temporary_ (1.5 0.0 -1.5);
    TABEL s_bias[2] _temporary_ (1.0 -1.0);

    GØR enc = 1 TIL 100;
        fi = 1 + floor(3 * rand('uniform'));
        si = 1 + floor(2 * rand('uniform'));
        facility  = facs[fi];
        specialty = specs[si];

        /* Reelt facilitet x speciale interaktionsled */
        interaction = 1.2 * f_bias[fi] * s_bias[si];

        satisfaction = 7.0 + f_bias[fi] + s_bias[si] + interaction
            + rand('normal', 0, 0.25);

        /* Hold på en 1-10 patientoplevelsesskala */
        HVIS satisfaction > 10 SÅ satisfaction = 10;
        HVIS satisfaction < 1  SÅ satisfaction = 1;
        satisfaction = round(satisfaction, 0.1);
        UDDATA;
    SLUT;

    BEHOLD facility specialty satisfaction;
KØR;



NOTE: DATA encounters


NOTE: Wrote encounters (100 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Trin 2 - Undersøg den rå vurderingsfordeling

Før modellering skal man bekræfte, at de syntetiske vurderinger opfører sig pænt, og gennemgå
gennemsnittene efter facilitet x speciale-celle. `PROC MEANS`-percentilnøgleord (`P25`,
`MEDIAN`, `P75`) opsummerer den overordnede spredning; det andet kald viser cellegennemsnittene,
der bærer den interaktion, som faktoriseringsmaskinen vil forsøge at genfinde.

In [2]:
PROCEDURE GENNEMSNIT data=encounters n mean std MIN p25 MEDIAN p75 MAX maxdec=2;
    VARIABEL satisfaction;
    MÆRKAT satisfaction='Tilfredshed';
KØR;

PROCEDURE GENNEMSNIT data=encounters mean NWAY maxdec=2;
    KLASSE facility specialty;
    VARIABEL satisfaction;
    MÆRKAT facility='Facilitet' specialty='Speciale' satisfaction='Tilfredshed';
KØR;


                                                  The MEANS Procedure

 Variable      Label               N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ---------------------------------------------------------------------------------------------------------------------------------
 satisfaction  Tilfredshed       100        6.75        1.81        4.40             5.60        6.20             8.00       10.00
 ---------------------------------------------------------------------------------------------------------------------------------

                                                  The MEANS Procedure

                                     Analysis Variable : satisfaction Tilfredshed

                                                                        N
                                    Facilitet           Speciale      Obs       Mean
                                    ------------------------------------------------
      


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 3 - Tilpas faktoriseringsmaskinen

Vi modellerer `satisfaction` som det intervalskalerede **mål** og de to kategoriske felter
`facility` og `specialty` som nominelle **input**-features. Vigtige valgmuligheder:

- `LEARN=0.02` - skridtstørrelsen for stokastisk gradientnedstigning. På dette lille,
  standardiserede design holder en moderat rate optimeringen stabil, samtidig med at
  cellestrukturen stadig tilpasses.
- `L2=0.0005` - let L2-regularisering, nok til at holde faktorerne fra at krympe for meget
  mod det samlede gennemsnit.
- `SEED=20240531` - reproducerbar initialisering.
- `OUT=scored` - skriv forudsigelser pr. række (`P_satisfaction`).
- `OUTSTAT=fitstats` - opsaml RMSE-historikken pr. iteration, så vi kan bekræfte konvergens.

`ID`-sætningen kopierer nøglefelterne over på den scorede output, så hver forudsigelse forbliver
mærket med sin facilitet og sit speciale.

In [3]:
PROCEDURE factmac data=encounters seed=20240531 learn=0.02 l2=0.0005
        out=scored OUTSTAT=fitstats;
    INDDATA facility specialty / level=nominal;
    target satisfaction / level=interval;
    id facility specialty;
    MÆRKAT facility='Facilitet' specialty='Speciale' satisfaction='Tilfredshed';
KØR;



                         The FACTMAC Procedure

  Target variable: Tilfredshed
  Input variable: Facilitet
  Input variable: Speciale

Fit Statistics

  L2 Regularization              0.0005
  Learning Rate                  0.02
  Max Iterations                 100
  Number of Factors              10
  Number of Features             2
  Number of Observations         100
  Train ASE                      1.561623
  Train MAE                      0.953557
  Train R-Square                 0.515847
  Train RASE                     1.249649

Variable Importance

  Variable                            Importance
  --------                            ----------
  SPECIALTY                             0.513485
  FACILITY                              0.486515




NOTE: PROC FACTMAC data=encounters

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC FACTMAC completed.


## Trin 4 - Bekræft konvergens

`OUTSTAT=`-tabellen registrerer trænings-RMSE ved hver SGD-iteration. En monotont faldende kurve,
der flader ud, fortæller os, at modellen er konvergeret inden for det (standard 100)
iterationsbudget.

In [4]:
PROCEDURE UDSKRIV data=fitstats(obs=15) MÆRKAT noobs;
KØR;



ITERATION          RMSE
---------  ------------
1          1.6784734207
2          1.2915242083
3          1.2679925124
4          1.2654232966
5          1.2650259551
6          1.2649577538
7          1.2649457032
8          1.2649435534
9          1.2649431684
10         1.2649430993
11         1.2649430869
12         1.2649430847
13         1.2649430843
14         1.2649430842
15         1.2649430842

... 85 more observations (showing 15 of 100)




NOTE: PROC PRINT data=fitstats

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 15 observations printed, 2 variables


## Trin 5 - Evaluer rekonstruktionsfejl

Den scorede tabel bærer den observerede `satisfaction` sammen med modellens `P_satisfaction`. Vi
udleder residualet og den absolutte fejl og opsummerer derefter. Et residual centreret nær nul og
en spredning af forudsagte vurderinger, der nærmer sig den observerede spredning (i stedet for
at falde sammen til en flad linje ved det samlede gennemsnit), indikerer, at faktoriserings-
maskinen reproducerer den indlejrede facilitet x speciale-struktur.

In [5]:
data resid;
    SÆT scored;
    residual = satisfaction - p_satisfaction;
    abs_err  = abs(residual);
KØR;

PROCEDURE UDSKRIV data=scored(obs=10) MÆRKAT noobs;
    MÆRKAT facility='Facilitet' specialty='Speciale' satisfaction='Tilfredshed';
KØR;

PROCEDURE GENNEMSNIT data=resid n mean std MIN p25 MEDIAN p75 MAX maxdec=3;
    VARIABEL satisfaction p_satisfaction residual abs_err;
    MÆRKAT satisfaction='Tilfredshed' p_satisfaction='Forudsagt tilfredshed'
          residual='Residual' abs_err='Absolut fejl';
KØR;


       Facilitet    Speciale  Tilfredshed  P_SATISFACTION
----------------  ----------  -----------  --------------
Sydklinikken      Onkologi            6.3    6.1577265381
Nordklinikken     Onkologi            5.4    6.0430846516
Centralklinikken  Kardiologi          7.9    9.5419769923
Sydklinikken      Kardiologi          4.7    5.8019161993
Centralklinikken  Onkologi            6.2    5.9284427651
Nordklinikken     Kardiologi           10    7.6719465958
Nordklinikken     Onkologi            5.9    6.0430846516
Nordklinikken     Kardiologi           10    7.6719465958
Sydklinikken      Kardiologi          4.9    5.8019161993
Centralklinikken  Onkologi            6.2    5.9284427651

... 90 more observations (showing 10 of 100)

                                                  The MEANS Procedure

 Variable        Label                         N        Mean     Std Dev     Minimum   Lower Quartile      Median   Upper Quartile     Maximum
 ------------------------------------------


NOTE: DATA resid


NOTE: Read 100 rows from scored.
NOTE: Wrote resid (100 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=scored

NOTE: PROC PRINT completed: 10 observations printed, 4 variables
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 6 - Fremhæv facilitet x speciale-performance

For kvalitetsforbedringsteams er den handlingsorienterede visning den forudsagte vurdering efter
**facilitet x speciale-kombination**. Kombinationer, hvis modelforudsagte oplevelse ligger et
godt stykke under systemgennemsnittet, er kandidater til gennemgang; kolonnen med absolut fejl
viser, hvor modellen passer rent, og hvor det klippede 1-10-loft begrænser, hvor højt en lineær
faktoriseringsmaskine kan nå.

In [6]:
PROCEDURE GENNEMSNIT data=resid mean NWAY maxdec=3;
    KLASSE facility specialty;
    VARIABEL p_satisfaction abs_err;
    MÆRKAT facility='Facilitet' specialty='Speciale'
          p_satisfaction='Forudsagt tilfredshed' abs_err='Absolut fejl';
KØR;


                                                  The MEANS Procedure

                                    Analysis Variable : p_satisfaction Forudsagt tilfredshed

                                                                        N
                                    Facilitet           Speciale      Obs       Mean
                                    ------------------------------------------------
                                    Centralklinikken    Kardiologi     13      9.542

                                                        Onkologi       20      5.928

                                    Nordklinikken       Kardiologi     18      7.672

                                                        Onkologi       16      6.043

                                    Sydklinikken        Kardiologi     20      5.802

                                                        Onkologi       13      6.158
                                    ----------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Fortolkning af resultaterne

Iterationshistorikken fra `OUTSTAT=` viser, at trænings-RMSE falder fra omkring 1,68 ved første
gennemløb til et plateau nær **1,265** ved omkring den syvende iteration, hvilket bekræfter, at
SGD-tilpasningen konvergerede fint inden for iterationsbudgettet. Fit Statistics-rapporten viser
en **trænings-R-Square på 0,516**, en **gennemsnitlig absolut fejl på 0,954** vurderingspoint og
en **RASE på 1,25** — faktoriseringsmaskinen forklarer omkring halvdelen af variansen i en
1-10-tilfredshedsscore, der har en standardafvigelse på 1,81, så den lærer reelt struktur i
stedet for blot at forudsige det samlede gennemsnit. Residualopsummeringen bekræfter dette:
residualgennemsnittet er praktisk talt nul (0,020), og de forudsagte vurderinger spænder fra
5,80 til 9,54 (standardafvigelse 1,27), hvilket følger — om end ikke fuldt matcher — den
observerede spredning.

Facilitet x speciale-tabellen omsætter den latente model til noget, et kvalitetsteam kan handle
på. Modellen rangerer `Centralklinikken`/`Kardiologi` højest (forudsagt 9,54) og
`Sydklinikken`/`Kardiologi` lavest (forudsagt 5,80), hvilket genfinder den indlejrede
interaktion, hvor kardiologi er fremragende på nogle sites og svag på andre. Kolonnen med
absolut fejl er ærlig om modellens begrænsninger: de to onkologi-celler reproduceres næsten
nøjagtigt (gennemsnitlig absolut fejl 0,19-0,24), mens `Nordklinikken`/`Kardiologi` under-
forudsiges (fejl 2,33), fordi dens sande vurderinger hober sig op ved det klippede 1-10-loft,
som en lineær faktoriseringsmaskine ikke kan nå.

**Næste skridt**, som en praktiker kunne tage: tilføje en `PARTITION`-holdout for at beskytte
mod overtilpasning, justere `LEARN=` og `L2=` for at afveje bias mod varians, eller udvide
feature-sættet (udbyder, besøgstype, sæson), så faktoriseringsmaskinen kan modellere
højere-ordens oplevelsesdrivere. På en fuldt licenseret installation ville et større
facilitet x speciale-gitter med flere besøg pr. celle lade modellen opløse finere
interaktionsstruktur, end det seks-cellede design vist her.